In [2]:
import pandas as pd 

df = pd.read_csv(
   "ml-100k/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
print(df.shape)
print(df["user_id"].nunique(), df["item_id"].nunique())

df = df.sort_values(["user_id","timestamp"])

df["rank_latest"] = df.groupby("user_id")["timestamp"].rank(method="first" , ascending=False)

test = df[df["rank_latest"] == 1]
val = df[df["rank_latest"] == 2]
train = df[df["rank_latest"]>2]

print(len(test),len(val),len(train))


(100000, 4)
943 1682
943 943 98114


In [3]:
import numpy as np

ranked_items = train["item_id"].value_counts().index.to_numpy()
seen = pd.concat([train, val]).groupby("user_id")["item_id"].apply(set).to_dict()

def popularity_ranker(user_id):
  return [i for i in ranked_items if i not in seen[user_id] ]

def evaluate(ranker, eval_df=test, K=10):
    hits, ndcgs = [], []
    for row in eval_df.itertuples():
        topk = ranker(row.user_id)[:K]
        if row.item_id in topk:
            pos = topk.index(row.item_id)          
            hits.append(1)
            ndcgs.append(1 / np.log2(pos + 2))     
        else:
            hits.append(0)
            ndcgs.append(0)
    return np.mean(hits), np.mean(ndcgs)

recall10, ndcg10 = evaluate(popularity_ranker)
print(f"Popularity  Recall@10: {recall10:.4f}   NDCG@10: {ndcg10:.4f}")

Popularity  Recall@10: 0.0859   NDCG@10: 0.0426
